# Harmony default - human_pancreas

Seeds: `0-9`

Metrics: BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR


In [1]:
import gc
import os
import sys

os.environ["PYTHONNOUSERSITE"] = "1"
sys.path = [
    path
    for path in sys.path
    if ".local/lib/python3.10/site-packages" not in path
]

PROJECT_ROOT = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics'
os.environ["R_HOME"] = f"{PROJECT_ROOT}/CondaEnv/harmony_optuna/lib/R"
sys.path.insert(0, f"{PROJECT_ROOT}/scib/")

import anndata as ad
import harmonypy as hm
import numpy as np
import pandas as pd
import scanpy as sc

from scib.metrics.metrics import metrics


In [2]:
DATASET_NAME = 'human_pancreas'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEY = 'tech'
LABEL_KEY = 'celltype'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

N_SEEDS = 10
N_PCS = 50
N_NEIGHBORS = 15
N_TOP_HVG = 2000

EMBED_KEY = "X_harmony"
CLUSTER_KEY = "leiden_opt"
METRIC_BATCH_KEY = "harmony_metric_batch"
METRIC_LABEL_KEY = "harmony_metric_label"

assert DATASET_NAME == 'human_pancreas'
assert BATCH_KEY == 'tech'
assert LABEL_KEY == 'celltype'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI', 'BRAS', 'KBET', 'CPCR')

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_key": BATCH_KEY,
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(range(N_SEEDS)),
    },
    name="configuration",
)


dataset                                         human_pancreas
data_path    /ubc/cs/research/beaver/projects/Single-cell-i...
batch_key                                                 tech
label_key                                             celltype
metrics      [BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR]
seeds                           [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Name: configuration, dtype: object

In [3]:
def validate_dataset_contents():
    adata_backed = ad.read_h5ad(DATA_PATH, backed="r")
    try:
        required_keys = (BATCH_KEY, LABEL_KEY)
        missing_keys = [
            key for key in required_keys if key not in adata_backed.obs
        ]
        if missing_keys:
            raise ValueError(
                f"Missing obs keys {missing_keys}. "
                f"Available keys: {list(adata_backed.obs.columns)}"
            )

        obs = adata_backed.obs[[BATCH_KEY, LABEL_KEY]].copy()
        pseudotime = None
        if "TRAJ" in SELECTED_METRICS:
            if "dpt_pseudotime" not in adata_backed.obs:
                raise ValueError(
                    "TRAJ requires adata.obs['dpt_pseudotime']."
                )
            pseudotime = adata_backed.obs["dpt_pseudotime"].copy()

        n_obs = int(adata_backed.n_obs)
        n_vars = int(adata_backed.n_vars)
    finally:
        adata_backed.file.close()

    if obs.isna().any().any():
        raise ValueError(
            "Missing batch/label values: "
            f"{obs.isna().sum().to_dict()}"
        )

    n_batches = int(obs[BATCH_KEY].nunique())
    n_labels = int(obs[LABEL_KEY].nunique())
    if n_batches < 2:
        raise ValueError(
            f"Batch metrics require at least 2 batches; found {n_batches}."
        )

    batch_coverage = obs.groupby(
        LABEL_KEY,
        observed=True,
    )[BATCH_KEY].nunique()
    label_counts = obs.groupby(LABEL_KEY, observed=True).size()

    if "TRAJ" in SELECTED_METRICS:
        finite_pseudotime = pseudotime.dropna()
        if (
            len(finite_pseudotime) < 2
            or finite_pseudotime.nunique() < 2
        ):
            raise ValueError(
                "TRAJ requires at least 2 usable pseudotime values."
            )

    if "ISOLISI" in SELECTED_METRICS:
        min_batch_coverage = int(batch_coverage.min())
        if min_batch_coverage >= n_batches:
            raise ValueError(
                "ISOLISI is enabled, but every label appears in all batches."
            )

    eligible_cpcr_labels = [
        str(label)
        for label in label_counts.index
        if int(label_counts.loc[label]) >= 100
        and int(batch_coverage.loc[label]) >= 2
    ]
    if "CPCR" in SELECTED_METRICS and not eligible_cpcr_labels:
        raise ValueError(
            "CPCR requires a label with at least 100 cells across "
            "at least 2 batches."
        )

    return {
        "n_obs": n_obs,
        "n_vars": n_vars,
        "n_batches": n_batches,
        "n_labels": n_labels,
        "cpcr_eligible_labels": len(eligible_cpcr_labels),
    }


validate_dataset_contents()


{'n_obs': 16382,
 'n_vars': 19093,
 'n_batches': 9,
 'n_labels': 14,
 'cpcr_eligible_labels': 9}

In [4]:
HARMONY_DEFAULTS = {
    "theta": None,
    "lamb": None,
    "sigma": 0.1,
    "nclust": None,
    "tau": 0.0,
    "block_size": 0.05,
    "max_iter_harmony": 10,
    "max_iter_kmeans": 20,
    "epsilon_cluster": 1e-5,
    "epsilon_harmony": 1e-4,
    "alpha": 0.2,
}


def run_harmony_default(adata, seed):
    adata_int = adata.copy()

    if "highly_variable" in adata_int.var:
        adata_int = adata_int[:, adata_int.var["highly_variable"]].copy()
    else:
        sc.pp.highly_variable_genes(
            adata_int,
            flavor="seurat",
            n_top_genes=N_TOP_HVG,
        )
        adata_int = adata_int[:, adata_int.var["highly_variable"]].copy()

    sc.tl.pca(
        adata_int,
        n_comps=N_PCS,
        svd_solver="arpack",
    )
    x_pca = np.asarray(adata_int.obsm["X_pca"], dtype=np.float32)

    harmony_result = hm.run_harmony(
        x_pca,
        adata_int.obs,
        BATCH_KEY,
        **HARMONY_DEFAULTS,
        verbose=False,
        random_state=seed,
        device="cpu",
    )

    x_harmony = np.asarray(harmony_result.Z_corr)
    if (
        x_harmony.shape[0] != adata_int.n_obs
        and x_harmony.shape[1] == adata_int.n_obs
    ):
        x_harmony = x_harmony.T
    if x_harmony.shape[0] != adata_int.n_obs:
        raise ValueError(
            f"Harmony output shape {x_harmony.shape} is incompatible "
            f"with n_obs={adata_int.n_obs}."
        )
    if not np.all(np.isfinite(x_harmony)):
        raise ValueError("Harmony produced NaN or Inf in X_harmony.")

    adata_int.obsm[EMBED_KEY] = x_harmony
    return adata_int


pd.Series(HARMONY_DEFAULTS, name="upstream_default_argument")


theta                    NaN
lamb                     NaN
sigma                0.10000
nclust                   NaN
tau                  0.00000
block_size           0.05000
max_iter_harmony    10.00000
max_iter_kmeans     20.00000
epsilon_cluster      0.00001
epsilon_harmony      0.00010
alpha                0.20000
Name: upstream_default_argument, dtype: float64

In [5]:
def metric_value(metric_series, key):
    if key not in metric_series.index:
        raise KeyError(
            f"Metric {key} not found. "
            f"Available metrics: {list(metric_series.index)}"
        )

    value = metric_series[key]
    if isinstance(value, (list, tuple, np.ndarray)):
        values = np.asarray(value, dtype=float)
        if values.size == 0 or not np.all(np.isfinite(values)):
            raise ValueError(
                f"Metric {key} returned non-finite values: {values}"
            )
        return float(np.mean(values))

    value = float(value)
    if not np.isfinite(value):
        raise ValueError(
            f"Metric {key} returned a non-finite value: {value}"
        )
    return value


def compute_joint_metrics(adata_raw, adata_int):
    sc.pp.neighbors(
        adata_int,
        use_rep=EMBED_KEY,
        n_neighbors=N_NEIGHBORS,
        random_state=0,
    )

    metric_key_map = {
        METRIC_BATCH_KEY: BATCH_KEY,
        METRIC_LABEL_KEY: LABEL_KEY,
    }
    for adata_name, adata_obj in (
        ("adata_raw", adata_raw),
        ("adata_int", adata_int),
    ):
        existing_aliases = [
            alias for alias in metric_key_map if alias in adata_obj.obs
        ]
        if existing_aliases:
            raise ValueError(
                f"Reserved metric obs keys already exist in {adata_name}: "
                f"{existing_aliases}"
            )
        for alias, source_key in metric_key_map.items():
            adata_obj.obs[alias] = adata_obj.obs[source_key].copy()

    metric_df = metrics(
        adata=adata_raw,
        adata_int=adata_int,
        batch_key=METRIC_BATCH_KEY,
        label_key=METRIC_LABEL_KEY,
        embed=EMBED_KEY,
        cluster_key=CLUSTER_KEY,
        verbose=False,
        bami_="BAMI" in SELECTED_METRICS,
        casw_="CASW" in SELECTED_METRICS,
        scgraphm_="SCGRAPHM" in SELECTED_METRICS,
        traj_="TRAJ" in SELECTED_METRICS,
        isolisi_="ISOLISI" in SELECTED_METRICS,
        bras_="BRAS" in SELECTED_METRICS,
        kbet_="KBET" in SELECTED_METRICS,
        cpcr_="CPCR" in SELECTED_METRICS,
    )
    metric_series = metric_df.iloc[:, 0]

    result = {
        key: metric_value(metric_series, key)
        for key in SELECTED_METRICS
    }
    bio_score = float(
        np.mean([result[key] for key in SELECTED_BIO_METRICS])
    )
    batch_score = float(
        np.mean([result[key] for key in SELECTED_BATCH_METRICS])
    )
    joint_score = float(0.6 * bio_score + 0.4 * batch_score)

    if not np.all(np.isfinite([bio_score, batch_score, joint_score])):
        raise ValueError(
            "Aggregation produced a non-finite score: "
            f"bio={bio_score}, batch={batch_score}, "
            f"joint={joint_score}"
        )

    result["bio_score"] = bio_score
    result["batch_score"] = batch_score
    result["joint_score"] = joint_score
    return result


In [6]:
results = []

for seed in range(N_SEEDS):
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_harmony_default(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()

results_df = pd.DataFrame(results)
results_df



===== seed 0 =====


2026-07-21 13:43:17,727 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...


2026-07-21 13:43:18,040 - harmonypy - INFO - KMeans initialization complete.


Cell type silhouette score...


Per cell type PC regression...


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.7546181065490183


optimised clustering against harmony_metric_label
optimal cluster resolution: 0.2
optimal score: 0.8046353206433532
BRAS...


seed           9.000000
BAMI           0.804635
CASW           0.619115
SCGRAPHM       0.622269
ISOLISI        0.867714
BRAS           0.838274
KBET           0.511046
CPCR           0.955801
bio_score      0.728433
batch_score    0.768374
joint_score    0.744409
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,ISOLISI,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,0,0.806583,0.618984,0.629793,0.809376,0.834670,0.508488,0.956217,0.716184,0.766458,0.736294
1,1,0.809610,0.619042,0.630397,0.653145,0.834741,0.510523,0.955612,0.678048,0.766959,0.713612
2,2,0.810286,0.619086,0.629212,0.720654,0.835465,0.511378,0.956247,0.694809,0.767697,0.723964
3,3,0.809325,0.619654,0.630394,0.876174,0.834062,0.509224,0.957341,0.733887,0.766876,0.747082
4,4,0.803672,0.619424,0.641005,0.856777,0.825517,0.487102,0.955867,0.730219,0.756162,0.740596
5,5,0.807941,0.618940,0.629693,0.791902,0.834936,0.508607,0.955739,0.712119,0.766428,0.733843
6,6,0.814615,0.619259,0.629871,0.876304,0.835117,0.509218,0.957182,0.735012,0.767172,0.747876
7,7,0.809009,0.619656,0.630425,0.863188,0.835753,0.510056,0.956650,0.730570,0.767486,0.745336
8,8,0.811666,0.619356,0.631229,0.788754,0.835506,0.509588,0.955994,0.712751,0.767029,0.734463
9,9,0.804635,0.619115,0.622269,0.867714,0.838274,0.511046,0.955801,0.728433,0.768374,0.744409


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.808734,0.003074,0.803672,0.814615
CASW,0.619251,0.000250,0.618940,0.619656
SCGRAPHM,0.630429,0.004264,0.622269,0.641005
ISOLISI,0.810399,0.071054,0.653145,0.876304
BRAS,0.834404,0.003150,0.825517,0.838274
KBET,0.507523,0.006869,0.487102,0.511378
CPCR,0.956265,0.000574,0.955612,0.957341
bio_score,0.717203,0.017751,0.678048,0.735012
batch_score,0.766064,0.003346,0.756162,0.768374
joint_score,0.736748,0.010442,0.713612,0.747876
